# Time-series momentum across assets

**Plain idea.** Long an asset after a year of positive excess return, short
after a year of negative, scaled to equal risk (Moskowitz, Ooi and Pedersen
2012, JFE 104(2)). The real-data study is `research/tsmom/tsmom.py`.

In [ ]:
import _paths  # noqa: F401  (puts research/ modules on sys.path)
import numpy as np
import pandas as pd
import tsmom

## Simulated assets with a planted trend

Four assets. Each has a slow AR(1) drift that persists for months, plus
noise; the trend is what TSMOM is meant to find.

In [ ]:
rng = np.random.default_rng(7)
months = pd.date_range("1980-01-31", periods=420, freq="ME")
drift = np.zeros((len(months), 4))
for t in range(1, len(months)):
    drift[t] = 0.95 * drift[t - 1] + rng.normal(0, 0.003, 4)
rx = pd.DataFrame(drift + rng.normal(0, 0.04, (len(months), 4)), index=months,
                  columns=["eq", "fx1", "fx2", "bond"])

## Task 1: the weight by hand

For asset `eq` at the last month, compute `w_hand` = sign of the trailing
12-month sum times `0.40 / sigma`, with `sigma = tsmom.ewma_vol(rx)` at the
same month.

In [ ]:
sigma = tsmom.ewma_vol(rx)
w_hand = ...
# Exercise: replace the ... above, then run the check cell below.

In [ ]:
assert abs(w_hand - tsmom.weights(rx)["eq"].iloc[-1]) < 1e-12

## Task 2: the portfolio

Run `tsmom.tsmom(rx)` into `res`; print the portfolio Sharpe and turnover.

In [ ]:
res = ...
# Exercise: replace the ... above, then run the check cell below.

In [ ]:
assert res["stats"]["TSMOM"]["sharpe_annual"] > 0
assert res["label"] == "exploratory"

**What to take away.** With a planted trend the strategy earns money. It
loses at turning points (see the worked example in `tsmom.py`), and on
real data most of its return came in a few long trends.